سلول ۱: آماده‌سازی مسیرها و تنظیمات

In [1]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

project_root = os.path.abspath("..")

if project_root not in sys.path:
    sys.path.append(project_root)

load_dotenv(Path(project_root) / ".env")

print("Project root:", project_root)

Project root: /Users/macbookpro/AMIR_DATA/00_Project/VSCode__project/RAG_finance


سلول ۲: لود کردن تنظیمات پروژه

In [2]:
import importlib
import config.settings as settings

importlib.reload(settings)

DATA_VECTORSTORE_PATH = settings.DATA_VECTORSTORE_PATH
COLLECTION_NAME = settings.COLLECTION_NAME

ARVAN_AI_API_KEY = os.getenv("ARVAN_AI_API_KEY")
ARVAN_EMBEDDING_URL = os.getenv("ARVAN_EMBEDDING_URL")

QDRANT_LOCAL_PATH = Path(DATA_VECTORSTORE_PATH) / "qdrant_local"

print("Vectorstore path:", QDRANT_LOCAL_PATH)
print("Collection name:", COLLECTION_NAME)
print("Arvan key loaded:", bool(ARVAN_AI_API_KEY))
print("Arvan URL loaded:", bool(ARVAN_EMBEDDING_URL))

Vectorstore path: /Users/macbookpro/AMIR_DATA/00_Project/VSCode__project/RAG_finance/data/vectorstore/qdrant_local
Collection name: financial_knowledge_base
Arvan key loaded: True
Arvan URL loaded: True


سلول ۳: اتصال به Qdrant Local

In [3]:
from qdrant_client import QdrantClient

client_qdrant = QdrantClient(
    path=str(QDRANT_LOCAL_PATH)
)

collection_info = client_qdrant.get_collection(
    collection_name=COLLECTION_NAME
)

print("Connected to Qdrant local")
print("Collection status:", collection_info.status)
print("Points count:", collection_info.points_count)
print("Vector size:", collection_info.config.params.vectors.size)
print("Distance:", collection_info.config.params.vectors.distance)

Connected to Qdrant local
Collection status: green
Points count: 168
Vector size: 3072
Distance: Cosine


سلول ۴: ساخت تابع Embedding برای Query

In [4]:
from openai import OpenAI

client_arvan = OpenAI(
    api_key=ARVAN_AI_API_KEY,
    base_url=ARVAN_EMBEDDING_URL
)

def embed_query(query: str):
    response = client_arvan.embeddings.create(
        model="text-embedding-3-large",
        input=query
    )
    
    return response.data[0].embedding

test_query = "سواد مالی یعنی چه؟"
test_vector = embed_query(test_query)

print("Query:", test_query)
print("Vector length:", len(test_vector))
print("First 5 values:", test_vector[:5])

Query: سواد مالی یعنی چه؟
Vector length: 3072
First 5 values: [-0.03668212890625, -0.006343841552734375, -0.0018835067749023438, -0.03985595703125, 0.01415252685546875]


سلول ۵: جستجو در Qdrant

In [5]:
from qdrant_client.models import QueryResponse

search_results = client_qdrant.query_points(
    collection_name=COLLECTION_NAME,
    query=test_vector,
    limit=5,
    with_payload=True
)

print("Number of results:", len(search_results.points))

for i, point in enumerate(search_results.points, start=1):
    payload = point.payload
    
    print("=" * 80)
    print("Rank:", i)
    print("Score:", point.score)
    print("Chunk ID:", payload.get("chunk_id"))
    print("Title:", payload.get("title"))
    print("URL:", payload.get("url"))
    print("\nText preview:")
    print(payload.get("text", "")[:700])

Number of results: 5
Rank: 1
Score: 0.5360664155751962
Chunk ID: 5C92BB0D_3
Title: فیل سواد مالی
URL: https://www.fintelligence.ir/Blog/Detail/5C92BB0D/%D9%81%DB%8C%D9%84-%D8%B3%D9%88%D8%A7%D8%AF-%D9%85%D8%A7%D9%84%DB%8C

Text preview:
یا سرمایه‌گذاری را تمام آن چیزی معرفی کنیم که او در آینده لازم دارد. روی سخن با کسانی نیست که در برندسازی از نام «سواد مالی» بهره خود را می‌برند؛ مخاطب آن‌هایی هستند که دغدغه تربیت نسل آینده را دارند. در کف هر یک اگر شمعی بدی / اختلاف از گفت‌شان بیرون شدی دانش، نور است و اگر پای تخصص در سواد مالی به میان آید، بسیاری از آسیب‌های برآمده از نگاه بخشی و انحرافی به سواد مالی رخت برمی‌بندد. این تعهدی است که در برابر آیندگان داریم.

داستان فیلِ در تاریکی، از مشهورترین داستان‌های مثنوی مولوی است؛ فیلی که در تاریکی، از ستون و ناودان گرفته تا تخت و بادبزن تلقی و برداشت می‌شود.

از نظرگه گفتشان شد مختلف / آن یکی دالش لقب داد این الف

حکایت سواد مالی و ما هم، حکایت فیل و مردمان هند است.

از آنجا که «سوا
Rank: 2
Score: 0.5257735149768874
Chunk ID: BD65A55B_1
Title: س

سلول ۶: تابع Retrieval تمیز

In [6]:
def retrieve_chunks(query: str, top_k: int = 5):
    query_vector = embed_query(query)

    results = client_qdrant.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,
        limit=top_k,
        with_payload=True
    )

    retrieved_chunks = []

    for point in results.points:
        payload = point.payload

        retrieved_chunks.append({
            "score": point.score,
            "chunk_id": payload.get("chunk_id"),
            "article_id": payload.get("article_id"),
            "title": payload.get("title"),
            "url": payload.get("url"),
            "text": payload.get("text"),
            "char_count": payload.get("char_count")
        })

    return retrieved_chunks

سلول ۷: تست تابع Retrieval

In [7]:
query = "سواد مالی یعنی چه؟"

results = retrieve_chunks(query, top_k=5)

print("Query:", query)
print("Results:", len(results))

for i, item in enumerate(results, start=1):
    print("=" * 80)
    print("Rank:", i)
    print("Score:", item["score"])
    print("Chunk ID:", item["chunk_id"])
    print("Title:", item["title"])
    print("URL:", item["url"])
    print("\nText preview:")
    print(item["text"][:700])

Query: سواد مالی یعنی چه؟
Results: 5
Rank: 1
Score: 0.5362950447866566
Chunk ID: 5C92BB0D_3
Title: فیل سواد مالی
URL: https://www.fintelligence.ir/Blog/Detail/5C92BB0D/%D9%81%DB%8C%D9%84-%D8%B3%D9%88%D8%A7%D8%AF-%D9%85%D8%A7%D9%84%DB%8C

Text preview:
یا سرمایه‌گذاری را تمام آن چیزی معرفی کنیم که او در آینده لازم دارد. روی سخن با کسانی نیست که در برندسازی از نام «سواد مالی» بهره خود را می‌برند؛ مخاطب آن‌هایی هستند که دغدغه تربیت نسل آینده را دارند. در کف هر یک اگر شمعی بدی / اختلاف از گفت‌شان بیرون شدی دانش، نور است و اگر پای تخصص در سواد مالی به میان آید، بسیاری از آسیب‌های برآمده از نگاه بخشی و انحرافی به سواد مالی رخت برمی‌بندد. این تعهدی است که در برابر آیندگان داریم.

داستان فیلِ در تاریکی، از مشهورترین داستان‌های مثنوی مولوی است؛ فیلی که در تاریکی، از ستون و ناودان گرفته تا تخت و بادبزن تلقی و برداشت می‌شود.

از نظرگه گفتشان شد مختلف / آن یکی دالش لقب داد این الف

حکایت سواد مالی و ما هم، حکایت فیل و مردمان هند است.

از آنجا که «سوا
Rank: 2
Score: 0.5261252410684141
Chunk ID: BD6

سلول ۸: ساخت Context از نتایج Retrieval

In [9]:
def build_context(retrieved_chunks):
    context_parts = []

    for i, item in enumerate(retrieved_chunks, start=1):
        part = f"""
[منبع {i}]
عنوان: {item["title"]}
آدرس: {item["url"]}
شناسه چانک: {item["chunk_id"]}

متن:
{item["text"]}
"""
        context_parts.append(part.strip())

    separator = "\n\n" + ("-" * 80) + "\n\n"
    return separator.join(context_parts)


context = build_context(results)

print("Context length:", len(context))
print(context[:3000])

Context length: 7255
[منبع 1]
عنوان: فیل سواد مالی
آدرس: https://www.fintelligence.ir/Blog/Detail/5C92BB0D/%D9%81%DB%8C%D9%84-%D8%B3%D9%88%D8%A7%D8%AF-%D9%85%D8%A7%D9%84%DB%8C
شناسه چانک: 5C92BB0D_3

متن:
یا سرمایه‌گذاری را تمام آن چیزی معرفی کنیم که او در آینده لازم دارد. روی سخن با کسانی نیست که در برندسازی از نام «سواد مالی» بهره خود را می‌برند؛ مخاطب آن‌هایی هستند که دغدغه تربیت نسل آینده را دارند. در کف هر یک اگر شمعی بدی / اختلاف از گفت‌شان بیرون شدی دانش، نور است و اگر پای تخصص در سواد مالی به میان آید، بسیاری از آسیب‌های برآمده از نگاه بخشی و انحرافی به سواد مالی رخت برمی‌بندد. این تعهدی است که در برابر آیندگان داریم.

داستان فیلِ در تاریکی، از مشهورترین داستان‌های مثنوی مولوی است؛ فیلی که در تاریکی، از ستون و ناودان گرفته تا تخت و بادبزن تلقی و برداشت می‌شود.

از نظرگه گفتشان شد مختلف / آن یکی دالش لقب داد این الف

حکایت سواد مالی و ما هم، حکایت فیل و مردمان هند است.

از آنجا که «سواد مالی» نام جذابی است، هر کسی که به جزئی از آن دسترسی دارد، نام کل را بر آن می‌گذارد. برخی، آن 

<div style="direction: rtl; white-space: normal; line-height: 1;">
عالی. این خروجی کاملاً تمیز شد. ✅

پس مرحله Retrieval تمام شد:

```text
✅ Query embedding با Arvan
✅ اتصال به Qdrant Local
✅ جستجوی top-k
✅ ساخت context برای RAG
```

حالا برویم مرحله بعد:

```text
07_generation.ipynb
```

هدفش:

```text
question + retrieved context → final answer
```

ولی قبلش یک نکته مهم: برای Generation نیاز به مدل پاسخ‌گو داریم، نه embedding.
یعنی باید ببینیم در آروان مدل chat هم فعال داری یا فقط Embedding-3-Large را خریدی.

</div>